# Fakultätszuordnung mit OpenAlex und GERiT

1. Schritt: Installieren der erforderlichen Python-Pakete:

In [1]:
import sys
#!{sys.executable} -m pip install pyalex
#!{sys.executable} -m pip install requests

In [2]:
import pyalex
import requests
import re

2. Schritt: OpenAlex API Key einfügen:

Die folgende Webseite erklärt, wie man einen OpenAlex Key erhalten kann: https://help.openalex.org/api/authentication/

In [9]:
OPENALEX_API_KEY = 'YOUR_API_KEY'
pyalex.config.api_key = OPENALEX_API_KEY

3. Schritt: Filter anpassen

Im folgenden Beispiel werden die OpenAlex IDs der Georg-August Unviersität Göttingen, der Staats- und Universitätsbibliothek Göttingen, der GWDG sowie der Universitätsmedizin Göttingen gelistet.

In [4]:
institution_ids = ['https://openalex.org/I74656192', # GAU
                   'https://openalex.org/I4387154063', # SUB
                   'https://openalex.org/I4210091733', # GWDG
                   'https://openalex.org/I4210116730' # UMG
                  ]

item_types = ['article', 'review']

publication_years = ['2025']

sample_size = 10

4. Schritt: Mapping herunterladen

Alle Mappings sind über folgenden Link einsehbar: https://github.com/naustica/lower_saxony_institutions/blob/main/data/mappings/README.md

In [5]:
gerit_gau_url = 'https://raw.githubusercontent.com/naustica/lower_saxony_institutions/main/data/mappings/gau.json'
response = requests.get(gerit_gau_url)
gerit_mapping = response.json()

In [6]:
def map_address(address: str, mapping_dict: dict):
    faculty = None
    faculty_id = None
    department = None
    department_id = None
    institute = None
    institute_id = None
    
    for k, v in mapping_dict.items():
        pattern = re.compile(r'|'.join(v.get('faculty_patterns')), re.IGNORECASE)
        res = bool(pattern.search(address))
        if res:
            faculty = k
            faculty_id = v.get('faculty_id')

    if faculty:
        for k, v in mapping_dict.get(faculty).get('suborganisations').items():
            pattern = re.compile(r'|'.join(v.get('department_patterns')), re.IGNORECASE)
            res = bool(pattern.search(address))
            if res:
                department = k
                department_id = v.get('department_id')

    if department:
        for k, v in mapping_dict.get(faculty).get('suborganisations').get(department).get('suborganisations').items():
            pattern = re.compile(r'|'.join(v.get('institute_patterns')), re.IGNORECASE)
            res = bool(pattern.search(address))
            if res:
                institute = k
                institute_id = v.get('institute_id')

    return dict(faculty=faculty, 
                faculty_id=faculty_id,
                department=department, 
                department_id=department_id, 
                institute=institute,
                institute_id=institute_id)

5. Schritt: OpenAlex API Anfrage erstellen

In [7]:
response = pyalex.Works().filter(
    authorships={'affiliations': {'institution_ids': '|'.join(institution_ids)}}, 
    type='|'.join(item_types), 
    publication_year='|'.join(publication_years)).sample(sample_size).get()

6. Schritt: API-Antwort auswerten

In [8]:
for article in response:

    openalex_id = article.get('id')
    
    authorships = article.get('authorships')

    for author in authorships:
        affiliations = author.get('affiliations')
        for affiliation in affiliations:
            if any(map(lambda id: id in institution_ids, affiliation.get('institution_ids'))):
                author_name = author.get('author').get('display_name')
                raw_affiliation_string = affiliation.get('raw_affiliation_string')
                mapping_dict = map_address(raw_affiliation_string, gerit_mapping)

                faculty = mapping_dict.get('faculty')
                faculty_id = mapping_dict.get('faculty_id')

                department = mapping_dict.get('department')
                department_id = mapping_dict.get('department_id')

                institute = mapping_dict.get('institute')
                institute_id = mapping_dict.get('institute_id')

                print(f'OpenAlex Works ID: {openalex_id}')
                
                print(f'Person: {author_name}')
                print(f'Affiliationsstring: {raw_affiliation_string}')
                print(f'Fakultät: {faculty}')
                print(f'Fakultät ID: {faculty_id}')
                print(f'Department: {department}')
                print(f'Department ID: {department_id}')
                print(f'Institut: {institute}')
                print(f'Institut ID: {institute_id}')

                print('#############################')

OpenAlex Works ID: https://openalex.org/W4409877402
Person: Smita Das
Affiliationsstring: Tropical Silviculture and Forest Ecology University of Göttingen  Gottingen Germany
Fakultät: Fakultät für Forstwissenschaften und Waldökologie
Fakultät ID: 13707
Department: Burckhardt-Institut
Department ID: 215713403
Institut: Abteilung Waldbau und Waldökologie der Tropen
Institut ID: 28029
#############################
OpenAlex Works ID: https://openalex.org/W4409877402
Person: Prakash Basnet
Affiliationsstring: Department for Spatial Structures and Digitization of Forests University of Göttingen  Gottingen Germany
Fakultät: Fakultät für Forstwissenschaften und Waldökologie
Fakultät ID: 13707
Department: Professur für Räumliche Strukturen und Digitalisierung von Wäldern
Department ID: 504088658
Institut: None
Institut ID: None
#############################
OpenAlex Works ID: https://openalex.org/W4409877402
Person: Dominik Seidel
Affiliationsstring: Centre of Biodiversity and Sustainable Land 